## 문제
- 데이터프레임에서 'Aspects'컬럼에 데이터들을 이용하여 분류 모델 생성하려 한다.
- SentimentText 텍스트를 이용하여 'Aspect', 'SentimentPolarity'의 값들을 예측
    1. df에서 'Aspects' 데이터를 추출
    2. SentimentText데이터를 문자형으로 이루어져있으니 학습에 대한 데이터의 형태로 변환(문자의 데이터를 숫자형 데이터) -> 토큰화(Okt), 벡터화(TF-IDF)
    3. 종속 변수는 'Aspect', 'SentimentPolarity'
    4. 분류 모델(LinearSVC) random_state만 42로 고정
    5. 테스트를 이용하여 분류가 잘되고 있는가? 정확도만 확인(197데이터를 로드하여 정확도 계산)
    - 벡터화, 모델링 파이프라인으로 연결해서 사용

In [164]:
test_text = [
    '색상이 마음에 든다',
    '설명에 비해 옷이 두껍진 않다',
    '길이가 너무 길지도 않고 짧지도 않다'
]

In [165]:
# 라이브러리 로드
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score

In [166]:
# 데이터 로드
df = pd.read_json("1-1.여성의류(196).json")
df.head(1)

,Index,RawText,Source,Domain,MainCategory,ProductName,Syllable,Word,GeneralPolarity,Aspects
0,1024338,안녕하세요 이웃님들 반갑습니다. 요즘 날씨가 많이 쌀쌀해졌죠? 요즘 계절에 입으면 ...,SNS,패션,여성의류,OO 경량 다운 자켓,513,121,1,"[{'Aspect': '디자인', 'SentimentText': '딱 기본 스타일인..."


In [167]:
# Aspects 컬럼의 데이터들을 이용하여 하나의 새로운 데이터프레임으로 생성
pd.DataFrame(df['Aspects'].sum())

,Aspect,SentimentText,SentimentWord,SentimentPolarity
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,11,1
1,두께,이것만 입기엔 얇지만,3,-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,12,1
3,색상,색상도 디자인도 무난해서,3,0
4,디자인,디자인도 무난해서,2,0
...,...,...,...,...
912,길이,길이감도 너무 짧거나 애매한 길이가 아니고 적당합니다.,7,1
913,활용성,트레이닝 세트나 데님 스커트 후드 원피스 등에도 잘 어울립니다.,9,1
914,디자인,전체적으로 고급스러움이 잘 녹아있는 디자인으로 되어 있고,7,1
915,품질,퀄리티도 넘 휼륭합니다.,3,1


In [168]:
# 각 행의 Aspects 데이터를 추출하여 리스트에 추가
new_list = []

# iterrows() : 데이터프레임에서 [(idx, df.loc[idx, ]), ...] 되돌려준다
for i, s in df.iterrows():
    # print(s['Aspects'])
    aspects = s['Aspects']
    for aspect in aspects:
        # aspect에서 'SentimentWord' 키는 제거
        # dict 형태의 데이터에서 특정 키를 제거 -> del
        try:
            del aspect['SentimentWord']
        except:
            pass
        # print(aspect)
        new_list.append(aspect)

aspect_df = pd.DataFrame(new_list)

In [169]:
# 분류 모델을 돌리기 위해 종속 변수들의 균형이 어떻게 되는가?
# polarity의 데이터의 개수들을 확인
aspect_df['SentimentPolarity'].value_counts()

SentimentPolarity
1     808
-1     81
0      28
Name: count, dtype: int64

In [170]:
aspect_df['Aspect'].value_counts()

Aspect
디자인     121
기능      101
소재       93
활용성      87
색상       79
핏        64
사이즈      64
가격       58
착용감      47
길이       35
무게       31
두께       28
촉감       26
품질       26
제품구성     25
신축성      21
마감       11
Name: count, dtype: int64

In [171]:
# 결측치가 존재하는가?
aspect_df.isna().sum()

Aspect               0
SentimentText        0
SentimentPolarity    0
dtype: int64

In [172]:
# 모든 value에게 좌우의 공백을 제거
aspect_df.iloc[:, :2] = aspect_df.iloc[:, :2].map(
    lambda x : x.strip()
)

In [173]:
# 빈 텍스트가 존재하는가?
aspect_df.isin(['']).sum()

Aspect               0
SentimentText        0
SentimentPolarity    0
dtype: int64

In [174]:
# Aspect 컬럼의 데이터들을 LabelEncoder로 변환
le = LabelEncoder()

aspect_df['Aspect'] = le.fit_transform(aspect_df['Aspect'])

In [175]:
aspect_df.head()

,Aspect,SentimentText,SentimentPolarity
0,4,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,1
1,3,이것만 입기엔 얇지만,-1
2,1,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,1
3,8,색상도 디자인도 무난해서,0
4,4,디자인도 무난해서,0


In [176]:
# Text를 토큰화 -> 벡터화 작업
from konlpy.tag import Okt

In [177]:
okt = Okt()

vectorizer = TfidfVectorizer(
    tokenizer=okt.morphs,
    lowercase=False,
    ngram_range=(1, 2)
)

In [178]:
# 분류 모델 정의
svc = LinearSVC(
    random_state=42
)

In [179]:
svc2 = LinearSVC(
    random_state=42
)

In [180]:
# Aspect 예측하기 위한 모델
pipe_aspect = Pipeline(
    [
        ('vector', vectorizer),
        ('clf', svc)
    ]
)
# Pola 예측하기 위한 모델
pipe_pola = Pipeline(
    [
        ('vector', vectorizer),
        ('clf', svc2)
    ]
)

In [181]:
X = aspect_df['SentimentText'].values
Y1 = aspect_df['Aspect'].values
Y2 = aspect_df['SentimentPolarity'].values

In [182]:
# 2개의 모델에 학습 -> 독립변수 Text, 종속변수 Aspect, Pola
# pipeline.fit() -> 스텝중 변환이 있으면 fit_transform() -> 모델에는 fit()
pipe_aspect.fit(X, Y1)
pipe_pola.fit(X, Y2)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(
c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<bound method...01D90785F230>>


In [183]:
# pipeline.predict() -> 스텝 중 변환이 있으면 transform() -> 모델에는 predict()
pred_aspect = pipe_aspect.predict(test_text)
pred_pola = pipe_pola.predict(test_text)

In [184]:
pred_pola

array(['1', '1', '1'], dtype=object)

In [185]:
pred_aspect

array([8, 4, 2])

In [186]:
le.inverse_transform(pred_aspect)

array(['색상', '디자인', '길이'], dtype=object)

In [187]:
df2 = aspect_df.copy()

In [188]:
# LabelEncoder를 원본으로 변환
df2['Aspect'] = le.inverse_transform(df2['Aspect'])

In [189]:
df2['target'] = df2['Aspect'] + '_' + df2['SentimentPolarity']

In [190]:
df2.head()

,Aspect,SentimentText,SentimentPolarity,target
0,디자인,딱 기본 스타일인데 또 입은 거 보면 깔끔하게 저렴해보이지 않는 디자인이라서,1,디자인_1
1,두께,이것만 입기엔 얇지만,-1,두께_-1
2,기능,초겨울까지는 운동 갈때 안에 얇은 기능성 반팔 입고 요것만 입어도 꽤 따뜻해요.,1,기능_1
3,색상,색상도 디자인도 무난해서,0,색상_0
4,디자인,디자인도 무난해서,0,디자인_0


In [191]:
# target 컬럼의 데이터를 LabelEncoder 변환
le2 = LabelEncoder()
df2['target'] = le2.fit_transform(df2['target'])

In [223]:
df2.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917 entries, 0 to 916
Data columns (total 4 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             917 non-null    object
 1   SentimentText      917 non-null    object
 2   SentimentPolarity  917 non-null    object
 3   target             917 non-null    int64 
dtypes: int64(1), object(3)
memory usage: 28.8+ KB


In [192]:
X_2 = df2['SentimentText']
Y_2 = df2['target']

In [193]:
pipe = Pipeline(
    [
        ('vector', vectorizer),
        ('clf', LinearSVC(random_state=42))
    ]
)

In [194]:
pipe.fit(X_2, Y_2)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('clf', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<bound method...01D90785F230>>


In [195]:
pipe.predict(test_text)

array([22, 25,  6])

In [213]:
# 종속변수가 2개인 경우 일반적으로 사용하는 객체
from sklearn.multioutput import MultiOutputClassifier

In [214]:
# 종속변수의 크기가 (2000, 2)
# 첫번쨰 종속데이터를 이용하여 fit() -> predict()
# 두번째 종속데이터를 이용하여 fit() -> predict()
# 위의 2개의 작업을 병렬로 처리

In [231]:
# 분류 모델을 생성
svc = LinearSVC(random_state=42)
# 멀티 아웃 모델을 생성
multi_model = MultiOutputClassifier(svc)

pipe_multi = Pipeline(
    [
        ('vector', vectorizer),
        ('model', multi_model)
    ]
)

In [232]:
aspect_df['SentimentPolarity'] = aspect_df['SentimentPolarity'].astype(int)

In [233]:
# 멀티 모델 종속은 2차원 그대로 사용
X = aspect_df['SentimentText'].values
Y1 = aspect_df[['Aspect', 'SentimentPolarity']].values

In [234]:
aspect_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 917 entries, 0 to 916
Data columns (total 3 columns):
 #   Column             Non-Null Count  Dtype 
---  ------             --------------  ----- 
 0   Aspect             917 non-null    int64 
 1   SentimentText      917 non-null    object
 2   SentimentPolarity  917 non-null    int64 
dtypes: int64(2), object(1)
memory usage: 21.6+ KB


In [235]:
Y1

array([[ 4,  1],
       [ 3, -1],
       [ 1,  1],
       ...,
       [ 4,  1],
       [14,  1],
       [ 4,  1]], shape=(917, 2))

In [236]:
pipe_multi.fit(X, Y1)

c:\Users\student\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\feature_extraction\text.py:517: UserWarning: The parameter 'token_pattern' will not be used since 'tokenizer' is not None'
  warnings.warn(


,steps,"[('vector', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,input,'content'
,encoding,'utf-8'
,decode_error,'strict'
,strip_accents,None
,lowercase,False
,preprocessor,None
,tokenizer,<bound method...01D90785F230>>


In [237]:
# 예측
pred_multi = pipe_multi.predict(test_text)

In [238]:
pred_aspect_origin = le.inverse_transform(pred_multi[:, 0])
pred_aspect_origin

array(['색상', '디자인', '길이'], dtype=object)

In [239]:
pred_pola = pred_multi[:, 1]
pred_pola

array([1, 1, 1])

In [240]:
# 문단인 장문의 데이터에서 문장별로 나눠주기
from konlpy.tag import Kkma

In [241]:
text = df.loc[0, 'RawText']

In [242]:
kkma = Kkma()

In [244]:
texts = kkma.sentences(text)

In [245]:
pred = pipe_multi.predict(texts)

In [246]:
le.inverse_transform(pred[:, 0])

array(['사이즈', '기능', '기능', '활용성', '디자인', '무게', '기능', '활용성', '활용성', '색상',
       '디자인', '활용성', '활용성', '디자인', '착용감'], dtype=object)

In [247]:
pred[:, 1]

array([1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1])